# MetaMARL tutorial: bilevel mechanism design in a shared fishery

This notebook introduces the main MetaMARL abstractions through the **fishery benchmark**.

We will build a two-level learning problem:

- a **society** of fishers learns how much to harvest and how much restoration effort to apply;
- a **regulator** learns a quota mechanism that intervenes on the fishers' harvest decisions;
- both levels interact through a shared fish-stock dynamical system.

The goal of the notebook is not to hide the configuration behind helper scripts. Instead, it shows how the pieces fit together so that you can replace the optimizer, mechanism, environment, reporter, or agent population without changing the overall workflow.

> **Rendering note.** Display equations in this notebook use standard Jupyter/MathJax delimiters (`$...$` and `$$...$$`). They are intentionally kept out of Python code blocks so the notebook renders them as mathematics.


## 1. The benchmark in one picture

At the society level, each fisher observes the shared resource and produces policy actions. The benchmark maps those actions through fishery mechanisms such as `Fishing` and `Restore`.

At the regulator level, a second optimizer controls an institutional mechanism such as `Quota`. The quota targets the fishers' harvest mechanism rather than replacing the fishers' policy.

Conceptually:

$$
\text{regulator parameters}
\;\longrightarrow\;
\text{institutional mechanism}
\;\longrightarrow\;
\text{society learning}
\;\longrightarrow\;
\text{fishery trajectory}
\;\longrightarrow\;
\text{regulator objective}.
$$

This is why MetaMARL exposes the two levels separately as:

```text
BilevelConfig
├── regulator(...)
└── society(...)
```


## 2. Fishery dynamics

Let $B_t$ denote fish-stock biomass, $K$ the carrying capacity, $r$ the intrinsic growth rate, and $p$ the Pella–Tomlinson shape parameter.

The benchmark advances the stock with the discrete-time update

$$
B_{t+1}
=
B_t
+
\frac{r}{p}B_t
\left[
1-\left(\frac{B_t}{K}\right)^p
\right]
+
\varepsilon_t
+
R_t
-
H_t,
$$

where:

- $\varepsilon_t$ is multiplicative process noise;
- $R_t$ is restoration added by the agents;
- $H_t$ is realized aggregate harvest.

When $p=1$, the biological growth term reduces to the Schaefer logistic form.

The environment also computes standard reference points:

$$
B_{\mathrm{MSY}}
=
K\left(\frac{1}{p+1}\right)^{1/p},
$$

$$
\mathrm{MSY}
=
\frac{rK}{(p+1)^{(p+1)/p}},
$$

and

$$
F_{\mathrm{MSY}}
=
\frac{\mathrm{MSY}}{B_{\mathrm{MSY}}}.
$$


### 2.1 Fisher actions

The policy does not directly emit biomass. Its raw action is decoded by a mechanism.

For harvesting, the benchmark squashes an unbounded scalar $z_t$ through

$$
h_t^{\mathrm{frac}}
=
\sigma\left(\frac{z_t}{4}\right),
$$

then scales it by the fisher's catch capacity,

$$
H_{i,t}
=
h_{i,t}^{\mathrm{frac}}
\left(
m F_{\mathrm{MSY}}
\frac{B_t}{N}
\right),
$$

where $m$ is `unregulated_f_multiplier` and $N$ is the number of fishers.

Choosing $m>1$ is important for the benchmark: it allows the unregulated society to request harvest above the sustainable reference level, so the institutional mechanism has a non-trivial problem to solve.

Restoration is decoded similarly:

$$
R_{i,t}
=
\rho K
\frac{r_{i,t}^{\mathrm{frac}}}{N},
$$

where $\rho$ is `restoration_effectiveness`.


## 3. The package mental model

There are four objects to keep separate.

**Environment.** `FisheryRegulatedEnv` owns the ecological state and transition dynamics.

**Agent configuration.** `AgentConfig` (and benchmark-specific subclasses such as `FishermanConfig`) describes who acts, which policy they use, how many copies exist, and which mechanisms belong to that agent.

**Mechanism.** A mechanism decodes or transforms a decision before it affects the system. In this benchmark:

- `Fishing` converts a policy action into harvest pressure;
- `Restore` converts a policy action into biomass restoration;
- `Quota` is owned by the regulator and targets the fishers' `"harvest"` mechanism.

**Optimizer configuration.** The society and regulator can use different optimizers. Here the regulator uses Evolution Strategies (ES), while the society uses APPO.

The important design point is that the bilevel container does **not** require these particular optimizer classes. It composes two optimizer configurations around a shared world.


## 4. Imports

The tutorial uses the benchmark objects already provided under `examples.bilevel_fishery`.

Unused experimental mechanisms are intentionally omitted from the imports so the minimal tutorial stays readable.


In [ ]:
import numpy as np
import ray
from gymnasium import spaces

from core.adaptors.ray.schema import RaySchema
from core.agents.base import AgentConfig
from core.callbacks import (
    log_and_report_episode_metrics,
    tag_episode_with_env_idx,
)
from core.mechanism.algorithms.quota import Quota
from core.optimizers.appo.config import APPOptimizerConfig
from core.optimizers.bilevel import BilevelConfig
from core.optimizers.es.config import ESConfig
from core.optimizers.es.schema import ESSchema
from core.reporting.wandb import WandbConfig

from examples.bilevel_fishery.metric_schema import FisheryMetricSchema
from examples.bilevel_fishery.queries import ES_QUERIES, INNER_QUERIES
from examples.bilevel_fishery.regulated_env import (
    FisheryRegulatedEnv,
    FishermanConfig,
    FishingConfig,
    RestoreConfig,
)
from examples.bilevel_fishery.regulator_env import FisheryRegulatorEnv


## 5. Shared experiment constants

Keeping the ecological constants in one place makes it harder for the regulator and society environments to silently disagree about the carrying capacity.

For a tutorial or smoke run, reduce the number of episodes and the horizon. For a real experiment, increase them again.


In [ ]:
K = 5_000
HORIZON = 100
NUM_FISHERS = 10
SEED = 42

ECOLOGY = {
    "r": 0.3,
    "K": K,
    "p": 1.0,
    "B0": 4_000,
    "fish_init": 4_000,
    "sigma": 0.02,
    "initial_stock_log_sigma": 0.05,
    "unregulated_f_multiplier": 2.0,
}

REGULATOR_ECOLOGY = {
    "sustainability_weight": 2,
    "sustainability_threshold": 0.20,
    "K": K,
}


## 6. Configure the regulator

The regulator is itself represented as an agent. Its policy controls a `Quota` mechanism.

The binding

```python
acts_on=("fisherman", "harvest")
```

means that the institutional mechanism targets the fishers' harvest mechanism. The regulator also receives the fish stock through

```python
obs_map={"resource_level": "fish"}
```

so the quota can depend on the current shared resource level.

The outer optimizer below is ES, but this is an optimizer choice rather than a property of the fishery benchmark.


In [ ]:
regulator_cfg = (
    ESConfig()
    .training(
        sigma=0.15,
        mean_lr=0.10,
        sigma_decay=1.0,
        sigma_lr=0.00,
        min_sigma=0.15,
        max_sigma=0.15,
        episodes=1_000,
    )
    .agents(
        AgentConfig(
            id="fisheries_regulator",
            policy_id="quota_policy",
            mechanisms=(
                Quota(
                    id="quota",
                    action_space=spaces.Box(
                        low=0.0,
                        high=1.0,
                        shape=(1,),
                        dtype=np.float32,
                    ),
                    acts_on=("fisherman", "harvest"),
                    obs_map={"resource_level": "fish"},
                    default=np.asarray(0.56224),
                ),
            ),
        ),
    )
    .environment(
        horizon=1,
        env=FisheryRegulatorEnv,
        env_config={
            "ecology_cfg": REGULATOR_ECOLOGY,
        },
    )
    .debugging(
        seed=SEED,
        num_seeds=1,
    )
    .reporting(
        schema=ESSchema,
        queries=ES_QUERIES,
    )
)


## 7. Configure the society

The society contains `NUM_FISHERS` copies of the same logical fisher agent.

Because `shared_policy=True`, all fishers share the policy identified by `"fisher_policy"`, while each environment instance still contains multiple fisher agents.

Each fisher owns two action mechanisms:

1. `Fishing(id="harvest")`
2. `Restore(id="restore")`

Each mechanism contributes one scalar action component, so the final policy action is assembled from the mechanism stack by the package.


In [ ]:
fisher = FishermanConfig(
    id="fisherman",
    policy_id="fisher_policy",
    shared_policy=True,
    count=NUM_FISHERS,
    mechanisms=(
        FishingConfig(
            id="harvest",
            action_space=spaces.Box(
                low=-np.inf,
                high=np.inf,
                shape=(1,),
                dtype=np.float32,
            ),
        ),
        RestoreConfig(
            id="restore",
            action_space=spaces.Box(
                low=-np.inf,
                high=np.inf,
                shape=(1,),
                dtype=np.float32,
            ),
        ),
    ),
    observation_space=spaces.Box(
        low=-np.inf,
        high=np.inf,
        shape=(5,),
        dtype=np.float32,
    ),
)


The APPO configuration below is the lower-level learning problem. The environment horizon and rollout length are deliberately aligned at `HORIZON` in this example.

The callback pair tags episodes with their environment index and forwards episode metrics into the reporting stack.


In [ ]:
society_cfg = (
    APPOptimizerConfig()
    .resources(
        num_cpus_for_main_process=1,
    )
    .framework(
        framework="torch",
    )
    .api_stack(
        enable_rl_module_and_learner=True,
        enable_env_runner_and_connector_v2=True,
    )
    .environment(
        env=FisheryRegulatedEnv,
        env_config={
            "ecology_cfg": ECOLOGY,
            "seed": 0,
        },
        horizon=HORIZON,
        disable_env_checking=False,
        schema=FisheryMetricSchema,
        queries=(),
    )
    .env_runners(
        num_env_runners=0,
        num_cpus_per_env_runner=1,
        num_gpus_per_env_runner=0,
        num_envs_per_env_runner=4,
        rollout_fragment_length=HORIZON,
        batch_mode="truncate_episodes",
        max_requests_in_flight_per_env_runner=1,
    )
    .learners(
        num_learners=0,
        num_gpus_per_learner=0,
    )
    .callbacks(
        on_episode_created=tag_episode_with_env_idx,
        on_episode_end=log_and_report_episode_metrics,
    )
    .training(
        episodes=50,
        vtrace=True,
        circular_buffer_num_batches=4,
        circular_buffer_iterations_per_batch=1,
        broadcast_interval=1,
        timeout_s_sampler_manager=10,
        timeout_s_aggregator_manager=10,
        gamma=0.99,
        lr=0.001,
        train_batch_size_per_learner=100,
        minibatch_size=100,
        num_epochs=1,
        entropy_coeff=0.001,
        grad_clip=40.0,
    )
    .evaluation(
        evaluation_interval=1,
        evaluation_duration=1,
        evaluation_duration_unit="episodes",
        evaluation_num_env_runners=0,
        evaluation_parallel_to_training=False,
        evaluation_config={
            "explore": False,
            "rollout_fragment_length": HORIZON,
            "batch_mode": "complete_episodes",
            "max_requests_in_flight_per_env_runner": 1,
        },
        base_seed=SEED,
        num_seeds=3,
    )
    .agents(
        fisher,
    )
    .fault_tolerance(
        restart_failed_env_runners=False,
    )
    .debugging(
        seed=SEED,
        num_seeds=1,
    )
    .reporting(
        min_time_s_per_iteration=0,
        min_sample_timesteps_per_iteration=0,
        min_train_timesteps_per_iteration=0,
        schema=RaySchema,
        queries=INNER_QUERIES,
    )
)


## 8. Compose the bilevel experiment

`BilevelConfig` is the top-level composition object.

The reporter belongs at this level because it receives information produced across the experiment. The regulator and society remain independently configured optimizer objects.


In [ ]:
bilevel_cfg = (
    BilevelConfig()
    .world(
        world_name="fishery_world",
    )
    .reporter(
        config=WandbConfig(
            project="bilevel",
            x_disable_stats=True,
            x_disable_meta=True,
            quiet=True,
            max_end_of_run_summary_metrics=0,
            max_end_of_run_history_metrics=0,
        )
    )
    .ray(
        device="cpu",
        num_cpus=4,
        omp_threads=1,
        logging_level="ERROR",
        runtime_env={
            "excludes": [
                "wandb/",
                ".git/",
                ".venv/",
                "__pycache__/",
                "ray_results/",
                "runs/",
            ]
        },
    )
    .regulator(
        regulator_cfg,
    )
    .society(
        society_cfg,
    )
)


## 9. Build and train

The current API has two explicit steps:

```python
optimizer = bilevel_cfg.build_optimizer()
optimizer.train()
```

Use `ray.shutdown()` before creating the optimizer when re-running notebook cells, and shut Ray down again at the end of the experiment. This avoids carrying a stale Ray runtime between notebook executions.


In [ ]:
ray.shutdown()

optimizer = bilevel_cfg.build_optimizer()

try:
    optimizer.train()
finally:
    ray.shutdown()


## 10. A smaller smoke configuration

The full configuration above is useful as documentation, but it is expensive for a first check.

For a smoke test, keep the same model structure and change only computational scale, for example:

- regulator `episodes`: a very small value;
- society `episodes`: a very small value;
- `NUM_FISHERS`: 2;
- `HORIZON`: 20;
- evaluation seeds: 1.

The important idea is to **shrink the experiment, not change its semantics**. This lets you verify construction, rollout, logging, and one complete bilevel update before starting a larger run.


## 11. How the fishery benchmark maps to MetaMARL

The benchmark is easier to extend when each responsibility stays in the right layer.

| Research concept | MetaMARL object | Fishery example |
|---|---|---|
| Shared dynamical system | Environment | `FisheryRegulatedEnv` |
| Individual decision maker | Agent config | `FishermanConfig` |
| Individual action semantics | Agent mechanism | `FishingConfig`, `RestoreConfig` |
| Institution / intervention | Regulator mechanism | `Quota` |
| Follower learning | Society optimizer | `APPOptimizerConfig` |
| Leader learning | Regulator optimizer | `ESConfig` |
| Coupling of both levels | Experiment config | `BilevelConfig` |
| Metrics | Typed schemas + queries | `FisheryMetricSchema`, `RaySchema`, `ESSchema` |

This separation is what makes the package reusable: replacing APPO, ES, the fishery dynamics, or the quota should not require rewriting the other layers.


## 12. Extending the tutorial

### Replace the society optimizer

Keep the environment and fisher mechanisms fixed, construct another optimizer config for the society, and pass it to:

```python
BilevelConfig().society(other_society_cfg)
```

### Replace the regulator optimizer

Keep the quota mechanism and regulator environment fixed, then replace `ESConfig` with another optimizer config supported by the package.

### Add another institutional mechanism

Add another mechanism to the regulator agent's `mechanisms=(...)` tuple. A restoration subsidy or a social-influence mechanism can therefore be composed with the quota without embedding that logic inside the fishery transition.

### Change the benchmark

Implement another environment and its agent/mechanism objects, then reuse the same regulator/society composition pattern.

The package boundary to preserve is:

$$
\text{environment dynamics}
\;\neq\;
\text{agent policy}
\;\neq\;
\text{mechanism intervention}
\;\neq\;
\text{optimizer}.
$$


## 13. Practical checks before a long run

Before launching a full experiment, verify that:

1. the regulator and society use the same carrying capacity $K$;
2. `rollout_fragment_length` is compatible with the society horizon;
3. the observation-space shape matches the observation emitted by `Fisherman`;
4. every mechanism has a unique `id` within its owning agent;
5. `acts_on=("fisherman", "harvest")` still matches the target agent and mechanism IDs;
6. a short run produces finite regulator fitness and finite fishery metrics;
7. Ray is cleanly shut down when a notebook run ends.

These checks catch configuration mismatches without coupling the tutorial to a particular optimizer implementation.


## 14. What to read next in the benchmark code

For a deeper implementation walkthrough, inspect these files in order:

1. `examples/bilevel_fishery/regulated_env.py` — fisher observations, rewards, harvesting/restoration mechanisms, and stock transition;
2. `examples/bilevel_fishery/regulator_env.py` — the regulator objective;
3. `examples/bilevel_fishery/metric_schema.py` — typed environment metrics;
4. `examples/bilevel_fishery/queries.py` — reporter queries;
5. the optimizer configs under `core.optimizers` — algorithm-specific settings.

The notebook should remain a **tutorial of the package composition**, while those modules remain the source of truth for benchmark implementation details.
